# 04 - Optimizer Selection & Tuning

This notebook performs a controlled comparison of **Adam**, **plain SGD**, and **SGD with momentum** using the exact lightweight Model B and the exact cached train/validation/test splits from Notebooks 01 and 03.

The experiment changes only the optimizer configuration. Every run receives the same initial model weights, seeded augmentation, batch order, epoch budget, loss, and metrics. The optimizer is selected using validation performance only; the untouched test set is evaluated after the choice is frozen.


In [ ]:
# Environment setup: Google Colab + shared Drive, with a local fallback.
import os

# Must be set before importing TensorFlow.
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/EN3150-A03-Resource-Constrained-CNN"
else:
    # Set EN3150_PROJECT_ROOT when running outside the repository/notebooks folder.
    REPO_DIR = os.path.abspath(os.environ.get("EN3150_PROJECT_ROOT", ".."))

print("IN_COLAB:", IN_COLAB)
print("REPO_DIR:", REPO_DIR)


## 1. Configuration and experiment protocol

Adam at $10^{-3}$ is the optimizer currently used by Notebooks 02 and 03. SGD and SGD with momentum use the same learning rate, $10^{-2}$, so their difference isolates the effect of momentum. Adam uses its standard suitable learning-rate scale; forcing all optimizer families to use one numerical learning rate would not be a meaningful comparison.

All three optimizers train for exactly 20 epochs. Early stopping and learning-rate schedules are intentionally absent because they would give different optimizers different effective training budgets.


In [ ]:
CONFIG = {
    "image_size": 64,
    "channels": 3,
    "num_classes": 6,
    "class_names": ["buildings", "forest", "glacier", "mountain", "sea", "street"],
    "dataset_name": "Intel Image Classification",
    "random_seed": 42,
    "batch_size": 32,
    "epochs": 20,
    "dropout_rate": 0.3,
    "model_b_max_trainable_params": 100_000,
    "convergence_fraction": 0.95,
}

OPTIMIZER_CONFIGS = {
    "Adam": {"learning_rate": 1e-3, "momentum": None},
    "SGD": {"learning_rate": 1e-2, "momentum": 0.0},
    "SGD + Momentum": {"learning_rate": 1e-2, "momentum": 0.9},
}

PROCESSED_SUBPATH = os.path.join("data", "processed", "intel-image-classification")
CANDIDATE_ROOTS = [
    REPO_DIR,
    os.path.join(REPO_DIR, "EN3150-A03-Resource-Constrained-CNN"),
]

PROCESSED_DIR = os.path.join(REPO_DIR, PROCESSED_SUBPATH)
for root in CANDIDATE_ROOTS:
    candidate = os.path.join(root, PROCESSED_SUBPATH)
    if os.path.exists(os.path.join(candidate, "X_train.npy")):
        PROCESSED_DIR = candidate
        break

CHECKPOINT_DIR = os.path.join(REPO_DIR, "models", "checkpoints", "optimizer_comparison")
TABLES_DIR = os.path.join(REPO_DIR, "results", "tables")
METRICS_DIR = os.path.join(REPO_DIR, "results", "metrics")
FIGURES_DIR = os.path.join(REPO_DIR, "results", "figures", "final")

for directory in (CHECKPOINT_DIR, TABLES_DIR, METRICS_DIR, FIGURES_DIR):
    os.makedirs(directory, exist_ok=True)

print("PROCESSED_DIR:", PROCESSED_DIR)
OPTIMIZER_CONFIGS


## 2. Imports and reproducibility


In [ ]:
import hashlib
import json
import platform
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from tensorflow import keras

SEED = CONFIG["random_seed"]
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError):
    pass

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


## 3. Load and validate the prepared data

No data is re-split here. Reusing Notebook 01's cached arrays prevents accidental split leakage and guarantees direct comparability with Notebooks 02 and 03.


In [ ]:
required_files = [
    "X_train.npy", "y_train.npy", "X_val.npy",
    "y_val.npy", "X_test.npy", "y_test.npy", "class_names.json",
]
missing = [name for name in required_files if not os.path.exists(os.path.join(PROCESSED_DIR, name))]
if missing:
    raise FileNotFoundError(
        f"Missing prepared data in {PROCESSED_DIR}: {missing}. "
        "Run 01_data_preparation.ipynb first."
    )

X_train = np.load(os.path.join(PROCESSED_DIR, "X_train.npy"), mmap_mode="r")
y_train = np.load(os.path.join(PROCESSED_DIR, "y_train.npy"), mmap_mode="r")
X_val = np.load(os.path.join(PROCESSED_DIR, "X_val.npy"), mmap_mode="r")
y_val = np.load(os.path.join(PROCESSED_DIR, "y_val.npy"), mmap_mode="r")
X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"), mmap_mode="r")
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"), mmap_mode="r")

with open(os.path.join(PROCESSED_DIR, "class_names.json"), encoding="utf-8") as file:
    class_names = json.load(file)

expected_shape = (CONFIG["image_size"], CONFIG["image_size"], CONFIG["channels"])
assert class_names == CONFIG["class_names"], "Class order differs from Notebook 01."
assert X_train.shape[1:] == expected_shape and X_val.shape[1:] == expected_shape
assert X_test.shape[1:] == expected_shape
assert len(X_train) == len(y_train) and len(X_val) == len(y_val) and len(X_test) == len(y_test)
assert X_train.dtype == np.float32 and X_val.dtype == np.float32 and X_test.dtype == np.float32
assert 0.0 <= float(X_train.min()) <= float(X_train.max()) <= 1.0
assert set(np.unique(y_train)).issubset(set(range(CONFIG["num_classes"])))

total_examples = len(X_train) + len(X_val) + len(X_test)
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "examples": [len(X_train), len(X_val), len(X_test)],
    "fraction": [len(X_train) / total_examples, len(X_val) / total_examples, len(X_test) / total_examples],
})
display(split_summary)
print("Image shape:", expected_shape, "| dtype:", X_train.dtype)


## 4. Deterministic input pipeline

Each run constructs a fresh training dataset with the same shuffle seed. `reshuffle_each_iteration=True` changes the order between epochs, but recreating the dataset with the same seed makes the epoch-by-epoch order repeat across optimizer runs.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def make_datasets():
    options = tf.data.Options()
    options.experimental_deterministic = True

    train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_ds = train_ds.shuffle(
        buffer_size=len(y_train),
        seed=SEED,
        reshuffle_each_iteration=True,
    )
    train_ds = train_ds.batch(CONFIG["batch_size"], drop_remainder=False)
    train_ds = train_ds.with_options(options).prefetch(AUTOTUNE)

    val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_ds = val_ds.batch(CONFIG["batch_size"]).with_options(options).prefetch(AUTOTUNE)

    test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
    test_ds = test_ds.batch(CONFIG["batch_size"]).with_options(options).prefetch(AUTOTUNE)
    return train_ds, val_ds, test_ds


## 5. Exact Model B architecture

Model B is copied from Notebook 03: a standard 32-filter stem followed by four depthwise-separable blocks, global average pooling, dropout, and a six-class softmax head. Using Model B keeps this three-run experiment computationally practical and tests the optimizer on the assignment's resource-constrained architecture.


In [ ]:
def separable_block(filters, pool=True):
    block = [
        keras.layers.SeparableConv2D(
            filters, kernel_size=3, padding="same", use_bias=False
        ),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
    ]
    if pool:
        block.append(keras.layers.MaxPooling2D(pool_size=2))
    return block


def build_model_b():
    layers = [
        keras.layers.Input(shape=(CONFIG["image_size"], CONFIG["image_size"], CONFIG["channels"])),
        keras.layers.RandomFlip("horizontal", seed=SEED + 1),
        keras.layers.RandomRotation(0.05, seed=SEED + 2),
        keras.layers.RandomZoom(0.1, seed=SEED + 3),
        keras.layers.Conv2D(32, kernel_size=3, padding="same", use_bias=False),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.MaxPooling2D(pool_size=2),
    ]
    layers += separable_block(64)
    layers += separable_block(128)
    layers += separable_block(128, pool=False)
    layers += separable_block(256)
    layers += [
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dropout(CONFIG["dropout_rate"], seed=SEED + 4),
        keras.layers.Dense(CONFIG["num_classes"], activation="softmax"),
    ]
    return keras.Sequential(layers, name="model_b_lightweight_cnn")


def trainable_parameter_count(model):
    return int(sum(np.prod(variable.shape) for variable in model.trainable_weights))


def weights_fingerprint(weights):
    digest = hashlib.sha256()
    for array in weights:
        digest.update(np.ascontiguousarray(array).tobytes())
    return digest.hexdigest()


keras.backend.clear_session()
keras.utils.set_random_seed(SEED)
reference_model = build_model_b()
initial_weights = [weight.copy() for weight in reference_model.get_weights()]
initial_fingerprint = weights_fingerprint(initial_weights)
total_params = int(reference_model.count_params())
trainable_params = trainable_parameter_count(reference_model)

assert trainable_params <= CONFIG["model_b_max_trainable_params"]
assert total_params == 67_398, "Architecture no longer matches Notebook 03 Model B."
assert trainable_params == 66_182, "Trainable parameter count no longer matches Notebook 03."

reference_model.summary()
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Initial-weight SHA-256:", initial_fingerprint)
del reference_model


## 6. Optimizers and training utilities

The checkpoint for each run stores the epoch with the lowest validation loss. Timing is wall-clock timing, so it is useful for comparing runs on the same runtime but should not be compared across different hardware sessions.


In [ ]:
def make_optimizer(name):
    cfg = OPTIMIZER_CONFIGS[name]
    if name == "Adam":
        return keras.optimizers.Adam(learning_rate=cfg["learning_rate"])
    return keras.optimizers.SGD(
        learning_rate=cfg["learning_rate"],
        momentum=cfg["momentum"],
    )


class EpochTimer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.times = []

    def on_epoch_begin(self, epoch, logs=None):
        self.start_time = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.times.append(time.perf_counter() - self.start_time)


def safe_name(name):
    return name.lower().replace(" + ", "_").replace(" ", "_")


def run_optimizer_experiment(name):
    print("\n" + "=" * 78)
    print(f"Training {name}: {OPTIMIZER_CONFIGS[name]}")
    print("=" * 78)

    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    model = build_model_b()
    model.set_weights(initial_weights)
    run_fingerprint = weights_fingerprint(model.get_weights())
    assert run_fingerprint == initial_fingerprint, "Run did not receive identical initial weights."

    train_ds, val_ds, _ = make_datasets()
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{safe_name(name)}_best.keras")
    checkpoint = keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1,
    )
    timer = EpochTimer()

    model.compile(
        optimizer=make_optimizer(name),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    started = time.perf_counter()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CONFIG["epochs"],
        callbacks=[checkpoint, timer],
        verbose=1,
    )
    total_seconds = time.perf_counter() - started
    history_dict = {
        key: [float(value) for value in values]
        for key, values in history.history.items()
    }

    best_loss_index = int(np.argmin(history_dict["val_loss"]))
    result = {
        "name": name,
        "optimizer_config": OPTIMIZER_CONFIGS[name].copy(),
        "initial_weights_sha256": run_fingerprint,
        "checkpoint_path": checkpoint_path,
        "history": history_dict,
        "epoch_times_sec": [float(value) for value in timer.times],
        "total_training_time_sec": float(total_seconds),
        "average_epoch_time_sec": float(np.mean(timer.times)),
        "best_epoch_by_val_loss": best_loss_index + 1,
        "best_val_loss": float(history_dict["val_loss"][best_loss_index]),
        "val_accuracy_at_best_loss": float(history_dict["val_accuracy"][best_loss_index]),
        "best_val_accuracy": float(max(history_dict["val_accuracy"])),
        "final_val_loss": float(history_dict["val_loss"][-1]),
        "final_val_accuracy": float(history_dict["val_accuracy"][-1]),
        "last_5_val_loss_std": float(np.std(history_dict["val_loss"][-5:])),
    }
    del model, train_ds, val_ds, history
    return result


## 7. Run the three controlled experiments


In [ ]:
results = {}
for optimizer_name in OPTIMIZER_CONFIGS:
    results[optimizer_name] = run_optimizer_experiment(optimizer_name)

assert len({result["initial_weights_sha256"] for result in results.values()}) == 1
print("\nAll runs began from the same weights:", initial_fingerprint)


## 8. Validation comparison and optimizer selection

The primary selection criterion is the **lowest best validation loss**, because the training objective is cross-entropy. If two values are equal to numerical precision, higher validation accuracy and then lower training time break the tie.

Convergence speed is defined reproducibly as the first epoch at which a run reaches 95% of the highest validation accuracy attained by any optimizer in this experiment. A missing value means the optimizer did not reach that shared target within 20 epochs.


In [ ]:
global_best_val_accuracy = max(result["best_val_accuracy"] for result in results.values())
convergence_target = CONFIG["convergence_fraction"] * global_best_val_accuracy

rows = []
for name, result in results.items():
    val_accuracy = np.asarray(result["history"]["val_accuracy"])
    reached = np.flatnonzero(val_accuracy >= convergence_target)
    convergence_epoch = int(reached[0] + 1) if len(reached) else None
    convergence_seconds = (
        float(sum(result["epoch_times_sec"][:convergence_epoch]))
        if convergence_epoch is not None else None
    )
    result["convergence_target_val_accuracy"] = float(convergence_target)
    result["convergence_epoch"] = convergence_epoch
    result["convergence_time_sec"] = convergence_seconds

    rows.append({
        "optimizer": name,
        "learning_rate": result["optimizer_config"]["learning_rate"],
        "momentum": result["optimizer_config"]["momentum"],
        "best_epoch_by_val_loss": result["best_epoch_by_val_loss"],
        "best_val_loss": result["best_val_loss"],
        "val_accuracy_at_best_loss": result["val_accuracy_at_best_loss"],
        "best_val_accuracy": result["best_val_accuracy"],
        "final_val_loss": result["final_val_loss"],
        "final_val_accuracy": result["final_val_accuracy"],
        "convergence_epoch": convergence_epoch,
        "convergence_time_sec": convergence_seconds,
        "average_epoch_time_sec": result["average_epoch_time_sec"],
        "total_training_time_sec": result["total_training_time_sec"],
        "last_5_val_loss_std": result["last_5_val_loss_std"],
    })

comparison_df = pd.DataFrame(rows)
comparison_df["_loss_tie"] = comparison_df["best_val_loss"].round(8)
comparison_df = comparison_df.sort_values(
    ["_loss_tie", "val_accuracy_at_best_loss", "total_training_time_sec"],
    ascending=[True, False, True],
).drop(columns="_loss_tie").reset_index(drop=True)

# Freeze the winner before inspecting the test set.
selected_optimizer = str(comparison_df.loc[0, "optimizer"])
for name in results:
    results[name]["selected_from_validation"] = (name == selected_optimizer)

print(f"Shared convergence target: {convergence_target:.4f} validation accuracy")
print("Selected from validation data only:", selected_optimizer)
display(comparison_df.round(5))


## 9. Test-set evaluation after selection

The optimizer choice above is now frozen. We evaluate each best-validation-loss checkpoint on the test set to complete the assignment comparison table, but these test values do not influence the selection.


In [ ]:
_, _, test_ds = make_datasets()
test_metrics = {}

for name, result in results.items():
    model = keras.models.load_model(result["checkpoint_path"])
    probabilities = model.predict(test_ds, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    truth = np.asarray(y_test)

    metrics = {
        "test_accuracy": float(accuracy_score(truth, predictions)),
        "test_precision_macro": float(precision_score(truth, predictions, average="macro", zero_division=0)),
        "test_recall_macro": float(recall_score(truth, predictions, average="macro", zero_division=0)),
        "confusion_matrix": confusion_matrix(
            truth, predictions, labels=range(CONFIG["num_classes"])
        ).tolist(),
    }
    result.update(metrics)
    test_metrics[name] = metrics
    del model, probabilities, predictions

for column in ("test_accuracy", "test_precision_macro", "test_recall_macro"):
    comparison_df[column] = comparison_df["optimizer"].map(
        {name: metrics[column] for name, metrics in test_metrics.items()}
    )

comparison_df["selected_from_validation"] = comparison_df["optimizer"].eq(selected_optimizer)
display(comparison_df.round(5))
print(
    f"Selected optimizer test accuracy (reported, not selected on): "
    f"{results[selected_optimizer]['test_accuracy']:.4f}"
)


## 10. Overlaid learning curves


In [ ]:
colors = {"Adam": "#1f77b4", "SGD": "#ff7f0e", "SGD + Momentum": "#2ca02c"}
figure, axes = plt.subplots(1, 2, figsize=(13, 4.8))

for name, result in results.items():
    epochs = np.arange(1, len(result["history"]["val_loss"]) + 1)
    width = 2.8 if name == selected_optimizer else 1.8
    label = f"{name}{' (selected)' if name == selected_optimizer else ''}"
    axes[0].plot(epochs, result["history"]["val_loss"], label=label, color=colors[name], linewidth=width)
    axes[1].plot(epochs, result["history"]["val_accuracy"], label=label, color=colors[name], linewidth=width)

axes[0].set(title="Validation loss", xlabel="Epoch", ylabel="Sparse categorical cross-entropy")
axes[1].set(title="Validation accuracy", xlabel="Epoch", ylabel="Accuracy")
axes[1].axhline(
    convergence_target, color="black", linestyle="--", linewidth=1,
    label=f"95% convergence target ({convergence_target:.3f})",
)
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()

figure.suptitle("Model B optimizer comparison (same initialization and data order)")
figure.tight_layout()
plot_path = os.path.join(FIGURES_DIR, "optimizer_comparison.png")
figure.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", plot_path)


## 11. Discussion: effect of momentum

Momentum accumulates a velocity (an exponentially decaying history of gradients). This usually damps oscillation in directions where gradients alternate and accelerates progress when gradients remain aligned. The code below reports the observed effect rather than assuming that momentum must win.


In [ ]:
sgd = results["SGD"]
momentum = results["SGD + Momentum"]

loss_delta = momentum["best_val_loss"] - sgd["best_val_loss"]
accuracy_delta = momentum["best_val_accuracy"] - sgd["best_val_accuracy"]
stability_delta = momentum["last_5_val_loss_std"] - sgd["last_5_val_loss_std"]

print("SGD vs. SGD + Momentum (momentum = 0.9, learning rate held at 0.01)")
print(f"Best validation-loss change: {loss_delta:+.5f} (negative favors momentum)")
print(f"Best validation-accuracy change: {accuracy_delta:+.5f} (positive favors momentum)")
print(f"Last-5 validation-loss std change: {stability_delta:+.5f} (negative is steadier)")

if momentum["convergence_epoch"] is None and sgd["convergence_epoch"] is None:
    print("Neither SGD variant reached the shared convergence target in the epoch budget.")
elif momentum["convergence_epoch"] is None:
    print("Only plain SGD reached the shared convergence target.")
elif sgd["convergence_epoch"] is None:
    print("Only SGD + Momentum reached the shared convergence target.")
else:
    epoch_change = momentum["convergence_epoch"] - sgd["convergence_epoch"]
    print(f"Convergence-epoch change: {epoch_change:+d} (negative means momentum converged sooner)")

if loss_delta < 0 and accuracy_delta >= 0:
    print("Observed conclusion: momentum improved both the best loss and best accuracy over plain SGD.")
elif loss_delta > 0 and accuracy_delta <= 0:
    print("Observed conclusion: momentum did not improve this SGD run under the tested settings.")
else:
    print("Observed conclusion: momentum produced a loss/accuracy trade-off; use the table and curves in the report.")


## 12. Save reproducible results


In [ ]:
table_path = os.path.join(TABLES_DIR, "optimizer_comparison.csv")
metrics_path = os.path.join(METRICS_DIR, "optimizer_comparison.json")
comparison_df.to_csv(table_path, index=False)

serializable_runs = {}
for name, result in results.items():
    serializable_runs[name] = {
        key: value
        for key, value in result.items()
        if key != "checkpoint_path"
    }
    serializable_runs[name]["checkpoint_path"] = os.path.relpath(result["checkpoint_path"], REPO_DIR)

artifact = {
    "experiment": "Model B optimizer comparison",
    "dataset_name": CONFIG["dataset_name"],
    "selection_rule": "lowest best validation loss; ties: higher validation accuracy, then lower time",
    "selected_optimizer": selected_optimizer,
    "test_set_used_for_selection": False,
    "convergence_definition": (
        f"first epoch reaching {CONFIG['convergence_fraction']:.0%} of the highest "
        "validation accuracy achieved by any run"
    ),
    "convergence_target_val_accuracy": float(convergence_target),
    "model": {
        "name": "Model B (Lightweight CNN)",
        "total_params": total_params,
        "trainable_params": trainable_params,
        "initial_weights_sha256": initial_fingerprint,
    },
    "config": CONFIG,
    "runs": serializable_runs,
    "comparison": [
        {
            key: (None if pd.isna(value) else value.item() if isinstance(value, np.generic) else value)
            for key, value in row.items()
        }
        for row in comparison_df.to_dict(orient="records")
    ],
    "software": {"python": platform.python_version(), "tensorflow": tf.__version__},
}

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(artifact, file, indent=2, allow_nan=False)

print("Saved table:", table_path)
print("Saved metrics:", metrics_path)
print("Saved plot:", plot_path)


## 13. Conclusion and integration decision

The following cell generates the final report-ready conclusion from the measured validation results. If Adam is selected, the existing optimizer setting in Notebooks 02 and 03 is supported. If another optimizer is selected, those notebooks should be updated and retrained with that optimizer before the final comparison notebook is run.


In [ ]:
winner = results[selected_optimizer]
print(
    f"Final optimizer choice: {selected_optimizer} "
    f"(learning rate={winner['optimizer_config']['learning_rate']}, "
    f"momentum={winner['optimizer_config']['momentum']})."
)
print(
    f"It was selected without test-set access because it achieved the lowest best "
    f"validation loss ({winner['best_val_loss']:.5f}) at epoch "
    f"{winner['best_epoch_by_val_loss']}, with validation accuracy "
    f"{winner['val_accuracy_at_best_loss']:.4f} at that checkpoint."
)
print(
    f"Its held-out test accuracy is {winner['test_accuracy']:.4f}; this value is "
    "reported only after the selection was frozen."
)

if selected_optimizer == "Adam":
    print("Decision: retain Adam at 1e-3 in Notebooks 02 and 03.")
else:
    print(
        f"Decision: replace Adam in Notebooks 02 and 03 with {selected_optimizer}, "
        "retrain both models, and regenerate their saved metrics before Notebook 08."
    )
